In [1]:
from datasets import load_dataset

/home/giridhar/miniconda3/envs/llm_learning/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
ds = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)

In [6]:
for d in ds:
    print(d)
    break

{'arxiv_id': '0902.3253', 'title': 'The gravitational wave background from star-massive black hole fly-bys', 'authors': ['Silvia Toonen', 'Clovis Hopman', 'Marc Freitag'], 'submission_date': '18 Feb 2009 (<a href="https://arxiv.org/abs/0902.3253v1">v1</a>), last revised 18 Jun 2009 (this version, v2)', 'comments': 'Accepted for publication by MNRAS; New appendix on population of phase space in presence of gravitational wave inspiral', 'primary_subject': 'Earth and Planetary Astrophysics (astro-ph.EP)', 'subjects': 'Earth and Planetary Astrophysics (astro-ph.EP)', 'doi': 'https://doi.org/10.48550/arXiv.0902.3253', 'abstract': 'Stars on eccentric orbits around a massive black hole (MBH) emit bursts of gravitational waves (GWs) at periapse. Such events may be directly resolvable in the Galactic centre. However, if the star does not spiral in, the emitted GWs are not resolvable for extra-galactic MBHs, but constitute a source of background noise. We estimate the power spectrum of this extr

In [8]:
type(d)

dict

In [9]:
import json

In [18]:
with open('../data/inputs/arxiv/arxiv_samples.jsonl', 'r') as f:
    cs_papers = [json.loads(line) for line in f.readlines()]

In [19]:
import pandas as pd
data = pd.DataFrame(cs_papers)

In [20]:
data['primary_subject'].value_counts()

primary_subject
Earth and Planetary Astrophysics (astro-ph.EP)                110
Astrophysics of Galaxies (astro-ph.GA)                        110
Solar and Stellar Astrophysics (astro-ph.SR)                  110
High Energy Astrophysical Phenomena (astro-ph.HE)             110
Instrumentation and Methods for Astrophysics (astro-ph.IM)    110
                                                             ... 
Applications (stat.AP)                                        110
Other Statistics (stat.OT)                                    110
Methodology (stat.ME)                                         110
Machine Learning (stat.ML)                                    110
Computation (stat.CO)                                         110
Name: count, Length: 148, dtype: int64

In [ ]:
# Download arXiv PDFs and extract text
from pathlib import Path
import json
import re
import requests
from tqdm import tqdm

try:
    from pypdf import PdfReader
except Exception as e:
    raise RuntimeError("Missing dependency: pypdf. Install with: pip install pypdf") from e

DATA_PATH = Path("../data/inputs/arxiv/arxiv_samples.jsonl")
OUT_DIR = Path("../outputs/arxiv")
PDF_DIR = OUT_DIR / "pdfs"
TEXT_OUT = OUT_DIR / "arxiv_texts.jsonl"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)

def pdf_url(arxiv_id: str) -> str:
    return f"https://arxiv.org/pdf/{arxiv_id}.pdf"

def clean_text(text: str) -> str:
    # Collapse excessive whitespace while preserving paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n", "\n\n", text)
    return text.strip()

def extract_text_from_pdf(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return clean_text("\n".join(parts))

def download_pdf(url: str, dest: Path) -> None:
    if dest.exists() and dest.stat().st_size > 0:
        return
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dest.write_bytes(r.content)

# Set a limit if you only want a subset
limit = None  # e.g., 50

with DATA_PATH.open() as f, TEXT_OUT.open("w") as out_f:
    for i, line in enumerate(tqdm(f, desc="Processing")):
        if limit is not None and i >= limit:
            break
        row = json.loads(line)
        arxiv_id = row.get("arxiv_id")
        if not arxiv_id:
            continue
        url = pdf_url(arxiv_id)
        pdf_path = PDF_DIR / f"{arxiv_id}.pdf"
        try:
            download_pdf(url, pdf_path)
            text = extract_text_from_pdf(pdf_path)
        except Exception as e:
            text = ""
            row["error"] = str(e)
        row_out = {
            "arxiv_id": arxiv_id,
            "title": row.get("title"),
            "pdf_url": url,
            "text": text,
        }
        out_f.write(json.dumps(row_out) + "\n")

print(f"Wrote: {TEXT_OUT}")
